In [ ]:
import os
import requests
import getpass
import numpy as np
import cv2
import matplotlib.pyplot as plt

In [ ]:
EUROPEANA_API_KEY = getpass.getpass("Enter your Europeana API key: ")

In [ ]:
search_url = "https://api.europeana.eu/record/v2/search.json"
params = {
    "query": "Vermeer",
    "limit": 10,  # Return 10 items
    "wsit": "true"  # Use WSIT profile
}

# Input API key in the header
headers = {
    "x-api-key": EUROPEANA_API_KEY
}

In [ ]:
try:
    response = requests.get(search_url, params=params, headers=headers)
    response.raise_for_status()

    # Get response as json
    data = response.json()

    # Show results
    print(f"\nTotal results number: {data.get('totalResults', 0)}")
    prev_url_list = []
    if data.get('items'):
        for i, item in enumerate(data['items'][:3], 1):  # Show the first 3 items
            print(f"\n--- item {i} ---")

            # artifact information
            for key in ['edmTitle', 'edmCreator', 'edmPreview']:
                value = item.get(key, item.get('title', 'N/A'))
                if value:
                    print(f"{key}: {value[:100]}..." if len(str(value)) > 100 else f"{key}: {value}")
                    if key == "edmPreview":
                        prev_url_list.append(value[0])

            # metadata
            print(f"dataProvider: {item.get('dataProvider', 'N/A')}")

            # previews
            preview = item.get('edmPreview', item.get('preview', 'N/A'))
            if preview and 'http' in preview:
                print(f"preview: {preview[:80]}...")

    else:
        print("There is no results")

except requests.exceptions.RequestException as e:
    print(f"\nError: {e}")
    if hasattr(e, 'response') and e.response:
        print(f"Status code: {e.response.status_code}")
        print(f"Response: {e.response.text[:200]}")
except Exception as e:
    print(f"\nError: {e}")

In [ ]:
prev_url_list

In [ ]:
resp = requests.get(prev_url_list[0], stream=True).raw
image = np.asarray(bytearray(resp.read()), dtype="uint8")
img = cv2.imdecode(image, cv2.IMREAD_COLOR)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
COLORS = ["blue","green","red"]

for idx, color in enumerate(COLORS):
    hist = cv2.calcHist([img], [idx], None, [256], [0, 256])
    plt.plot(hist, color=color)

plt.show()

In [ ]:
edges = cv2.Canny(img, threshold1=100, threshold2=200)
plt.imshow(edges, cmap='gray')
plt.title('Edge Detection')
plt.show()

In [ ]:
def get_dominant_colors(img, k=5):
    # 1. 画像を(画素数, 3)の形に変換し、浮動小数点型にする
    data = img.reshape((-1, 3)).astype(np.float32)

    # 2. K-meansの停止条件（10回反復するか、精度が1.0に達するまで）
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)

    # 3. K-means実行
    _, labels, centers = cv2.kmeans(data, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

    # 4. 各クラスターの画素数を数えて割合を出す
    _, counts = np.unique(labels, return_counts=True)

    # 5. 中心の色を整数型に戻す
    dominant_colors = centers.astype(np.uint8)

    return dominant_colors, counts

# 画像読み込み（前回のコードを流用）
# img = ... 

# 色を5つ抽出
colors, counts = get_dominant_colors(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), k=5)

# 結果を表示（カラーパレット作成）
palette = np.zeros((50, 300, 3), dtype=np.uint8)
start = 0
for selected_color, count in zip(colors, counts):
    end = start + int((count / sum(counts)) * 300)
    palette[:, start:end] = selected_color
    start = end

plt.imshow(palette)
plt.axis('off')
plt.title("Dominant Color Palette")
plt.show()